# Eyewear brand localization / OCR

This notebook runs the standalone `eyewear-localization/` pipeline using zero-shot **Florence-2 OCR** (or EasyOCR) and native class-agnostic SAM3 eyewear localization. Brand names are never sent to SAM3.

## 1. Clone, validate CUDA, and install

The setup deliberately uses Kaggle's preinstalled CUDA/Torch pair instead of `uv sync`: resolving a new CUDA wheel inside the worker can make SAM3 fail with `no kernel image is available`. It installs only missing lightweight packages, authenticates `HF_TOKEN`, downloads the gated SAM3 checkpoint, and fails loudly if the GPU or checkpoint is unavailable.


In [ ]:
import atexit
import importlib.util
import json, os, shutil, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

REPO_URL = "https://github.com/fez-Ox/pxModel-Object-Counting.git"
REPO_DIR = Path("/kaggle/working/pxModel-localization")
APP_DIR = REPO_DIR / "eyewear-localization"
SAM3_APP = REPO_DIR / "sam3-verbose-counting"
SAM3_CHECKPOINT = SAM3_APP / "checkpoints" / "sam3.pt"
PYTHON = sys.executable  # use Kaggle's GPU Python; do not replace its CUDA Torch with uv sync

# The Kaggle kernel push contains this notebook, while the inference package is
# cloned at a known public revision. The local runner verifies that origin/main
# matches the code that was pushed before it starts the kernel.
if not (APP_DIR / "infer.py").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# Kaggle's default image currently exposes a Tesla P100 (sm_60), while its
# newest preinstalled Torch build may only contain sm_70+ kernels. Detect that
# mismatch and install a known CUDA 12.4 Torch pair that still includes sm_60.
capability_probe = subprocess.run(
    [PYTHON, "-c", (
        "import torch; "
        "assert torch.cuda.is_available(), 'CUDA is unavailable'; "
        "cap=torch.cuda.get_device_capability(0); "
        "print(f'{cap[0]},{cap[1]}')"
    )],
    capture_output=True, text=True,
)
if capability_probe.returncode:
    raise RuntimeError(
        "Kaggle CUDA is unavailable before setup. stderr="
        f"{capability_probe.stderr.strip()}"
    )
gpu_capability = tuple(int(value) for value in capability_probe.stdout.strip().splitlines()[-1].split(','))
print("Kaggle GPU capability:", gpu_capability)
if gpu_capability < (7, 0):
    print("Installing P100-compatible PyTorch 2.5.1/cu124...")
    subprocess.run([
        PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall", "--no-deps",
        "torch==2.5.1", "torchvision==0.20.1",
        "--index-url", "https://download.pytorch.org/whl/cu124",
    ], check=True)

torch_probe = subprocess.run(
    [PYTHON, "-c", (
        "import torch; "
        "assert torch.cuda.is_available(), 'CUDA is unavailable after setup'; "
        "x=torch.randn(8, 8, device='cuda'); _=(x @ x).sum(); torch.cuda.synchronize(); "
        "print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), torch.cuda.get_arch_list())"
    )],
    capture_output=True, text=True,
)
if torch_probe.returncode:
    raise RuntimeError(
        "Kaggle CUDA/Torch preflight failed. Do not continue with an empty "
        f"localizer. stderr={torch_probe.stderr.strip()}"
    )
print("Kaggle GPU/Torch:", torch_probe.stdout.strip())

# Florence-2 remote code is incompatible with newer Kaggle Transformers
# builds (notably the _supports_sdpa initialization change). The project
# pins the compatible range; enforce its known-good version even when
# Kaggle already provides a newer package. This does not replace CUDA/Torch.
subprocess.run([
    PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir",
    "--no-deps", "--force-reinstall",
    "transformers==4.44.2", "tokenizers==0.19.1",
    "huggingface_hub==0.36.2",
], check=True)
print("Transformers pinned for Florence-2: 4.44.2")
# Install independent OCR candidates for the bounded comparison pass.
# RapidOCR uses bundled PP-OCR ONNX models and does not replace Torch;
# EasyOCR remains available as the existing GPU baseline.
subprocess.run([
    PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir",
    "rapidocr_onnxruntime==1.2.3", "easyocr==1.7.2",
], check=True)
print("OCR comparison backends ready: RapidOCR + EasyOCR + Florence-2")
# Try the higher-accuracy PP-OCRv5/PaddleOCR stack when this Kaggle
# Python version has compatible wheels. This candidate is optional: a
# failed install must not compromise the Torch/SAM3 run.
paddle_install = subprocess.run([
    PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir",
    "paddlepaddle==3.2.2", "paddleocr==3.2.0",
], capture_output=True, text=True)
if paddle_install.returncode:
    print("PaddleOCR unavailable on this worker; continuing with RapidOCR/EasyOCR.")
    print(paddle_install.stderr[-1000:])
else:
    print("OCR comparison backend ready: PaddleOCR PP-OCRv5")

# Install only lightweight Python packages that are absent. In particular, do
# not install torch/torchvision: Kaggle's preinstalled pair matches its GPU.
module_to_package = {
    "transformers": "transformers",
    "huggingface_hub": "huggingface_hub",
    "timm": "timm",
    "einops": "einops",
    "ftfy": "ftfy",
    "wrapt": "wrapt",
    "yaml": "PyYAML",
    "PIL": "Pillow",
}
missing = [package for module, package in module_to_package.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing non-CUDA packages:", missing)
    subprocess.run([PYTHON, "-m", "pip", "install", "-q", *missing], check=True)

# The runner injects the approved local token only into a temporary notebook
# copy. Prefer that injected value over a possibly stale Kaggle Secret; normal
# interactive Kaggle runs still use the HF_TOKEN Secret when no fallback exists.
HF_TOKEN_FALLBACK = ""

def get_hf_token():
    fallback = HF_TOKEN_FALLBACK.strip()
    if fallback:
        return fallback
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token:
        return token.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret("HF_TOKEN")
        if secret:
            return secret.strip()
    except Exception:
        pass
    return ""

def check_sam3_access(token):
    if not token:
        raise RuntimeError(
            "No HF_TOKEN available. Add an approved Kaggle Secret named HF_TOKEN "
            "or run scripts/run_kaggle.py with ~/.kaggle/hf_token."
        )
    request = urllib.request.Request(
        "https://huggingface.co/facebook/sam3/resolve/main/sam3.pt?download=true",
        headers={"Authorization": f"Bearer {token}", "Range": "bytes=0-0"},
    )
    try:
        with urllib.request.urlopen(request, timeout=45) as response:
            response.read(1)
            if response.status not in (200, 206):
                raise RuntimeError(f"Hugging Face SAM3 access returned HTTP {response.status}")
            print("Hugging Face SAM3 access: authenticated")
    except urllib.error.HTTPError as exc:
        raise RuntimeError(
            f"Hugging Face SAM3 access returned HTTP {exc.code}. "
            "The token is missing, expired, or lacks facebook/sam3 approval."
        ) from exc

hf_token = get_hf_token()
if not SAM3_CHECKPOINT.exists():
    check_sam3_access(hf_token)
    download_env = os.environ.copy()
    download_env["HF_TOKEN"] = hf_token
    download_command = [
        PYTHON, str(SAM3_APP / "download_model.py"),
        "--output", str(SAM3_CHECKPOINT), "--timeout", "600",
    ]
    # Pass the token through the environment, not command-line arguments.
    subprocess.run(download_command, cwd=SAM3_APP, env=download_env, check=True)
else:
    print("Using existing checkpoint:", SAM3_CHECKPOINT)

if not SAM3_CHECKPOINT.exists() or SAM3_CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("SAM3 checkpoint is missing or empty after authenticated download.")
print("SAM3 checkpoint ready:", SAM3_CHECKPOINT, "bytes=", SAM3_CHECKPOINT.stat().st_size)

def _cleanup_sam3_checkpoint():
    # The checkpoint is an input artifact, not a result. Remove it before
    # Kaggle packages the working directory so output download stays small.
    try:
        if SAM3_CHECKPOINT.exists():
            SAM3_CHECKPOINT.unlink()
            print("Removed temporary SAM3 checkpoint before kernel finalization.")
    except Exception as exc:
        print("Checkpoint cleanup warning:", exc)

atexit.register(_cleanup_sam3_checkpoint)
print("Inference Python:", PYTHON)

## 2. Select an image & parameters

Edit `IMAGE_PATH` or `TEST_IMAGES` to point to your retail display image. Edit the `TUNE` cell before running the OCR/full-attribution cells.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
INPUT_DIR = Path("/kaggle/input")
BRAND_FILE = APP_DIR / "brands.txt"
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OCR_CACHE_DIR = Path("/kaggle/working/ocr_cache")
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Collect all images from uploaded datasets or fallback
TEST_IMAGES = sorted([p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS])
if not TEST_IMAGES:
    sample_path = APP_DIR / "sample_display.jpg"
    if not sample_path.exists():
        from PIL import Image, ImageDraw, ImageFont
        img = Image.new("RGB", (1000, 700), color=(240, 240, 240))
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("DejaVuSans.ttf", 36)
        except Exception:
            font = ImageFont.load_default()
        draw.rectangle((350, 20, 650, 70), fill=(20, 20, 20))
        draw.text((400, 25), "GUCCI", fill=(255, 255, 255), font=font)
        draw.rectangle((380, 150, 480, 200), fill=(80, 80, 80))
        draw.rectangle((520, 150, 620, 200), fill=(80, 80, 80))
        img.save(sample_path)
    TEST_IMAGES = [sample_path]

# Full SAM3 validation defaults to one selected image to keep a GPU run
# observable and bounded. Set MAX_IMAGES=None to process the complete set.
TARGET_IMAGE_NAME = None
MAX_IMAGES = 1
IMAGE_PATH = next((p for p in TEST_IMAGES if p.name == TARGET_IMAGE_NAME), TEST_IMAGES[0])
try:
    probe = subprocess.run(
        [PYTHON, "-c", "import torch; print('cuda' if torch.cuda.is_available() else 'cpu')"],
        cwd=APP_DIR, capture_output=True, text=True, check=True,
    )
    DEVICE = probe.stdout.strip()
except Exception:
    DEVICE = "cpu"

print(f"Found {len(TEST_IMAGES)} test images to process.")
print("First image:", IMAGE_PATH)
print("Device:", DEVICE)


In [ ]:
# ─── Tune pipeline parameters ────────────────────────────────────────
# Set any of these to override defaults.

TUNE = {
    # --- OCR backend ---
    "ocr_backend": "paddleocr+florence2", # PP-OCR + bounded Florence; also easyocr, paddleocr, tesseract, none
    "ocr_scale": 2.0,           # upscale factor before OCR
    "ocr_fallback_budget": 6,   # max Florence-2 calls/image after unmatched Tesseract OCR
    "run_ocr_only": True,        # precompute stronger OCR before loading SAM3

    # --- Decision cascade & thresholds ---
    "tau": None,              # min probability threshold
    "margin": None,           # min probability margin
    "max_per_evidence": None, # cap on evidence score (default None)

    # --- C1 (on-product OCR) ---
    "c1_margin": None,        # crop margin fraction (default 0.25)
    "c1_scales": "2.0",       # one focused pass in the bounded full run
    "c1_no_sharpen": False,    # retain crop sharpening

    # --- Scene filter ---
    "person_threshold": None, # SAM3 score for person detection
    "poster_threshold": None, # SAM3 score for poster/ad detection
    "shelf_threshold": None,  # SAM3 score for shelf detection
    "shelf_filter": True,     # False to disable scene filtering

    # --- Free-form overrides (dotted key=value strings) ---
    "set": [
        # "fusion.tau=0.5",
    ],
}


def _tune_cli_args():
    """Convert TUNE dict into CLI flags for infer.py."""
    args = []
    ocr_backend = TUNE.get("ocr_backend", "tesseract+florence2")
    if ocr_backend:
        args += ["--ocr-backend", str(ocr_backend)]
    ocr_scale = TUNE.get("ocr_scale")
    if ocr_scale is not None:
        args += ["--ocr-scale", str(ocr_scale)]
    ocr_fallback_budget = TUNE.get("ocr_fallback_budget")
    if ocr_fallback_budget is not None:
        args += ["--ocr-fallback-budget", str(ocr_fallback_budget)]
    _map = {
        "tau": "--tau", "margin": "--margin",
        "max_per_evidence": "--max-per-evidence",
        "person_threshold": "--person-threshold",
        "poster_threshold": "--poster-threshold",
        "shelf_threshold": "--shelf-threshold",
    }
    for key, flag in _map.items():
        val = TUNE.get(key)
        if val is not None:
            args += [flag, str(val)]
    c1_margin = TUNE.get("c1_margin")
    if c1_margin is not None:
        args += ["--c1-margin", str(c1_margin)]
    c1_scales = TUNE.get("c1_scales")
    if c1_scales is not None:
        args += ["--c1-scales", str(c1_scales)]
    if TUNE.get("c1_no_sharpen", False):
        args.append("--c1-no-sharpen")
    for item in TUNE.get("set", []):
        args += ["--set", str(item)]
    if TUNE.get("shelf_filter") is False:
        args.append("--no-shelf-filter")
    return args

## 3. OCR-only test

This optional stage tests OCR independently without needing SAM3. It is disabled by default because the full pass below runs the same OCR once.

In [ ]:
def run_pipeline(checkpoint=None, images=None, out_dir=None, ocr_cache_dir=None, backend=None, visualize=True):
    selected_images = list(images) if images is not None else [IMAGE_PATH]
    effective_out = Path(out_dir) if out_dir is not None else OUTPUT_DIR
    effective_out.mkdir(parents=True, exist_ok=True)
    command = [
        PYTHON, "infer.py", *[str(path) for path in selected_images],
        "--brand-file", str(BRAND_FILE),
        "--device", DEVICE,
        "--no-vlm-audit",
        "--out", str(effective_out),
    ]
    tune_args = _tune_cli_args()
    if backend is not None and "--ocr-backend" in tune_args:
        tune_args[tune_args.index("--ocr-backend") + 1] = str(backend)
    command += tune_args
    if ocr_cache_dir is not None:
        command += ["--ocr-cache-dir", str(ocr_cache_dir)]
    if not visualize:
        command.append("--no-visualization")
    if checkpoint is not None:
        command += ["--sam3-checkpoint", str(checkpoint)]
    print("Running:", " ".join(command[:4]), "...", flush=True)
    completed = subprocess.run(command, cwd=APP_DIR, text=True)
    if completed.returncode:
        completed.check_returncode()
    results = {}
    for image_path in selected_images:
        result_path = effective_out / f"{image_path.stem}.json"
        if not result_path.exists():
            raise RuntimeError(f"Missing result for {image_path}: {result_path}")
        results[image_path.stem] = json.loads(result_path.read_text())
    if len(results) == 1:
        print("Effective config:", next(iter(results.values())).get("effective_config", {}))
    else:
        print("Processed images:", sorted(results))
    return results


ocr_result = {"text_detections": [], "signs": []}
if TUNE.get("run_ocr_only", False):
    print("OCR cache will be precomputed immediately before the full pass.")

In [ ]:
for sign in ocr_result["signs"]:
    scope = sign.get("scope", {})
    region = scope.get("region_bbox", [])
    print(f"  {sign['brand']} ({sign['text']}) -> {scope.get('type', '?')}")
    print(f"    region={region}  scope_conf={scope.get('confidence', 0):.2f}")

## 4. Full attribution test with SAM3 & Precision Cascade

Runs class-agnostic eyewear localization + Florence-2 OCR C1 + C2 signage scope + Precision Cascade.

In [ ]:
if not SAM3_CHECKPOINT.exists():
    raise RuntimeError("SAM3 checkpoint is unavailable; refusing to run a false zero-detection full pass.")
selected_full_images = (
    TEST_IMAGES if MAX_IMAGES is None else [IMAGE_PATH] + [
        path for path in TEST_IMAGES if path != IMAGE_PATH
    ][:max(0, int(MAX_IMAGES) - 1)]
)
if TUNE.get("run_ocr_only", False):
    print(f"\n=== OCR CACHE PASS ({TUNE.get("ocr_backend", "default")}) ===")
    ocr_results = run_pipeline(
        images=selected_full_images,
        out_dir=OCR_CACHE_DIR,
        visualize=False,
    )
    ocr_result = ocr_results[IMAGE_PATH.stem]
    for name, result in sorted(ocr_results.items()):
        print(f"{name}: OCR signs={[sign['brand'] for sign in result.get('signs', [])]}")
full_results = run_pipeline(
    SAM3_CHECKPOINT,
    selected_full_images,
    ocr_cache_dir=OCR_CACHE_DIR if TUNE.get("run_ocr_only", False) else None,
    # Reuse cached scene OCR, but retain Florence-2 for bounded C1 crop
    # calls after SAM3 releases its image features.
    backend=TUNE.get("ocr_backend") if TUNE.get("run_ocr_only", False) else None,
)
bad_results = {}
for name, result in full_results.items():
    reasons = []
    if result.get("backends", {}).get("localizer", {}).get("name") != "sam3-class-agnostic":
        reasons.append("SAM3 backend unavailable")
    if not result.get("instances"):
        reasons.append("no retained instances")
    unresolved = [
        item["instance_id"] for item in result.get("outputs", [])
        if item.get("brand") in (None, "unknown")
    ]
    if unresolved:
        reasons.append(f"unresolved instances: {unresolved}")
    if not result.get("evidence"):
        reasons.append("no attribution evidence")
    if reasons:
        bad_results[name] = reasons
if bad_results:
    raise RuntimeError("Full attribution validation failed: " + str(bad_results))
full_result = full_results[IMAGE_PATH.stem]
for name, result in sorted(full_results.items()):
    decisions = [item.get("brand", "unknown") for item in result.get("outputs", [])]
    print(f"{name}: instances={len(result.get('instances', []))} signs={len(result.get('signs', []))} "
          f"outputs={len(decisions)} brands={decisions}")
print("\nPer-instance decisions (Precision Cascade):")
for output in full_result.get("outputs", []):
    path = output.get("decision_path", "none")
    product_b = output.get("product_brand")
    zone_b = output.get("zone_brand")
    print(
        f"  {output['instance_id']} -> final: {output['brand']} (via {path}) "
        f"[product={product_b}, zone={zone_b}]"
    )
print(f"\nExcluded by scene filter: {len(full_result.get('excluded_instances', []))}")
for excluded in full_result.get("excluded_instances", []):
    print("  ", excluded["instance_id"], excluded["reasons"])

## 5. C2 column-boundary diagnostics

Inspect the inferred column boundaries and per-instance evidence.

In [ ]:
print("=" * 72)
print("C2 COLUMN DIAGNOSTICS")
print("=" * 72)

columns_seen = {}
for ev in full_result.get("evidence", []):
    if ev["cue"] != "C2":
        continue
    support = ev.get("support", {})
    col_left = support.get("column_left")
    col_right = support.get("column_right")
    if col_left is not None and col_right is not None:
        key = (round(col_left, 1), round(col_right, 1))
        columns_seen[key] = ev["brand"]

if columns_seen:
    print("\nInferred columns (from C2 evidence):")
    for (left, right), brand in sorted(columns_seen.items()):
        print(f"  [{left:.0f} – {right:.0f}] -> {brand}")
else:
    print("\n(No column info in C2 evidence — C2 may not have found header signs)")

print("\nPer-instance C2 assignment:")
c2_by_instance = {}
for ev in full_result.get("evidence", []):
    if ev["cue"] == "C2":
        c2_by_instance.setdefault(ev["instance_id"], []).append(ev)

for inst in full_result.get("instances", []):
    cx = inst["centroid"][0] if inst.get("centroid") else inst["bbox"][0] + inst["bbox"][2] / 2
    evs = c2_by_instance.get(inst["id"], [])
    if evs:
        brands = ", ".join(f"{e['brand']}({e['confidence']:.2f})" for e in evs)
        print(f"  {inst['id']} cx={cx:.0f} -> {brands}")
    else:
        print(f"  {inst['id']} cx={cx:.0f} -> (no C2 evidence)")

c1_evs = [e for e in full_result.get("evidence", []) if e["cue"] == "C1"]
if c1_evs:
    print("\nC1 on-product detections:")
    for e in c1_evs:
        s = e.get("support", {})
        print(f"  {e['instance_id']} -> {e['brand']} conf={e['confidence']:.2f} "
              f"text={s.get('raw_text', '?')!r}")
else:
    print("\n(No C1 evidence — no on-product text detected)")

## 6. Save annotated results

The cell writes an annotated JPEG to the output directory without rendering the high-resolution image in the notebook.


In [ ]:
from PIL import Image, ImageDraw, ImageFont

img = Image.open(IMAGE_PATH).convert("RGB")
draw = ImageDraw.Draw(img, "RGBA")
try:
    font = ImageFont.truetype("DejaVuSans.ttf", size=max(18, img.width // 50))
except Exception:
    font = ImageFont.load_default()

COLORS = [
    (255, 100, 100, 200),
    (100, 255, 100, 200),
    (100, 100, 255, 200),
    (255, 200, 50, 200),
    (200, 100, 255, 200),
    (100, 255, 255, 200),
]

if columns_seen:
    boundaries = set()
    for (left, right), brand in sorted(columns_seen.items()):
        boundaries.add(left)
        boundaries.add(right)
    for bx in sorted(boundaries):
        if 5 < bx < img.width - 5:
            for y in range(0, img.height, 20):
                draw.line([(bx, y), (bx, min(y + 10, img.height))],
                          fill=(255, 255, 0, 180), width=3)

    for i, ((left, right), brand) in enumerate(sorted(columns_seen.items())):
        color = COLORS[i % len(COLORS)]
        mid_x = (left + right) / 2
        draw.text((mid_x - 30, 10), brand.upper(), fill=color, font=font)

outputs_by_id = {o["instance_id"]: o for o in full_result.get("outputs", [])}
for inst in full_result.get("instances", []):
    x, y, w, h = inst["bbox"]
    decision = outputs_by_id.get(inst["id"], {})
    brand = decision.get("brand", "unknown")
    path = decision.get("decision_path")
    color = (35, 180, 75, 230) if brand != "unknown" else (200, 100, 30, 200)
    draw.rectangle((x, y, x + w, y + h), outline=color, width=3)
    label = f"{inst['id']}: {brand}"
    if path and path != "none":
        label += f" ({path})"
    elif decision.get("abstained"):
        label += " (abstain)"
    text_box = draw.textbbox((x, y), label, font=font)
    draw.rectangle(text_box, fill=color)
    draw.text((x, y), label, fill=(255, 255, 255), font=font)

annotated_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_columns.jpg"
img.save(annotated_path)
print("Annotated image saved to:", annotated_path)


## 7. Inspect the JSON contract

The result keeps raw OCR, scoped signs, cue evidence, and final decisions separate for auditing.

In [ ]:
from pprint import pprint
pprint({
    "signs": full_result["signs"],
    "evidence": full_result["evidence"],
    "outputs": full_result["outputs"],
})

## 8. Cleanup

The checkpoint is removed automatically so it is not uploaded as a notebook result.


In [ ]:
_cleanup_sam3_checkpoint()
print("Final output files:", sorted(str(p) for p in OUTPUT_DIR.glob("*")))